# Library

In [2]:
import torch
import copy
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import torch.backends.cudnn as cudnn
import numpy as np
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
import time
import os
from PIL import Image
from tempfile import TemporaryDirectory
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision.transforms import v2
from PIL import Image, ImageOps
from torchvision.transforms import functional as F
from torchvision.models import MobileNet_V2_Weights

# Dataset

In [3]:
class CustomImageDataset(Dataset):
    def __init__(self, img_dir, transform=None, target_transform=None):
        self.img_dir = img_dir
        self.image_filenames = []
        self.transform = transform
        self.target_transform = target_transform

        for f in os.listdir(img_dir):
            try:
                # Lấy nhãn từ tên file, giả sử tên là "5_abc.jpg"
                label = int(f.split('_')[0])
                if 0 <= label <= 9:
                    self.image_filenames.append(f)
            except:
                continue  # Bỏ qua file không hợp lệ

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        img_name = self.image_filenames[idx]
        img_path = os.path.join(self.img_dir, img_name)

        # Load ảnh và giữ nguyên RGB
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        label = int(img_name.split('_')[0])  # đảm bảo phù hợp lại lần nữa
        if self.target_transform:
            label = self.target_transform(label)

        return image, label

In [ ]:
for f in os.listdir('E:/data/train'):
    try:
        label = int(f.split('_')[0])
        if label > 9:
            print(f"Lỗi nhãn >9: {f}")
    except:
        print(f"Không đọc được nhãn từ: {f}")

In [ ]:
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((256, 256)),  # đảm bảo ảnh đều trước khi crop
        transforms.RandomEqualize(p=0.5),
        transforms.RandomRotation(10),  # chống nghiêng nhẹ
        transforms.RandomAffine(
            degrees=0,
            translate=(0.1, 0.1),  # dịch chữ đi chút xíu
            scale=(0.9, 1.1),      # phóng to/thu nhỏ nhẹ
        ),
        transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),  # crop ngẫu nhiên
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]) 
    ]),

    'val': transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ]),
}

# Dataset & Dataloader
train_dataset = CustomImageDataset('E:/data/train', transform=data_transforms['train'])
val_dataset = CustomImageDataset('E:/data/val', transform=data_transforms['val'])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=8)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=8)

dataloaders = {'train': train_loader, 'val': val_loader}
dataset_sizes = {'train': len(train_dataset), 'val': len(val_dataset)}

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


# Train model

In [ ]:
model = models.mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)

for param in model.features.parameters():
    param.requires_grad = True

model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.last_channel, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, 10)
)

model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=1e-2, momentum=0.9)
scheduler = lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

In [ ]:
def train_model(model, criterion, optimizer, scheduler, dataloaders, dataset_sizes, device, num_epochs=10):
    since = time.time()
    best_model_path = 'best_model.pth'  # File lưu mô hình tốt nhất

    torch.save(model.state_dict(), best_model_path)  # Lưu tạm thời trước
    best_acc = 0.0

    model.to(device)

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 10)

        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()
            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]
            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # Lưu mô hình tốt nhất theo val accuracy
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                torch.save(model.state_dict(), best_model_path)
                print(f'>> SAVED best model (epoch {epoch}, acc {epoch_acc:.4f})')

        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:.4f}')

    # Khôi phục model tốt nhất
    model.load_state_dict(torch.load(best_model_path))
    return model

In [ ]:
model = train_model(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    dataloaders=dataloaders,
    dataset_sizes=dataset_sizes,
    device=device,
    num_epochs=10
)

Epoch 0/9
----------
train Loss: 1.1512 Acc: 0.6027
val Loss: 0.5464 Acc: 0.8499
>> SAVED best model (epoch 0, acc 0.8499)

Epoch 1/9
----------
train Loss: 0.4876 Acc: 0.8711
val Loss: 0.2818 Acc: 0.9450
>> SAVED best model (epoch 1, acc 0.9450)

Epoch 2/9
----------
train Loss: 0.3484 Acc: 0.9039
val Loss: 0.3422 Acc: 0.9056

Epoch 3/9
----------
train Loss: 0.3022 Acc: 0.9226
val Loss: 0.2881 Acc: 0.9281

Epoch 4/9
----------
train Loss: 0.2912 Acc: 0.9234
val Loss: 0.1775 Acc: 0.9625
>> SAVED best model (epoch 4, acc 0.9625)

Epoch 5/9
----------
train Loss: 0.1751 Acc: 0.9527
val Loss: 0.1410 Acc: 0.9656
>> SAVED best model (epoch 5, acc 0.9656)

Epoch 6/9
----------
train Loss: 0.1363 Acc: 0.9635
val Loss: 0.1230 Acc: 0.9725
>> SAVED best model (epoch 6, acc 0.9725)

Epoch 7/9
----------
train Loss: 0.1257 Acc: 0.9640
val Loss: 0.1065 Acc: 0.9681

Epoch 8/9
----------
train Loss: 0.1044 Acc: 0.9682
val Loss: 0.1620 Acc: 0.9681

Epoch 9/9
----------
train Loss: 0.1139 Acc: 0.9657
